# Outline of the main script

In [2]:

#imports:

from chime.calibration import load_Learmonth_data
from scipy import interpolate

import numpy as np
import pandas as pd

import glob
from tqdm import trange
import os
from datetime import datetime


#placing the files in here:

learmonth = '/home/scratch/dbautist/CHIME_archive/learmonthData/'
learmonth_files = glob.glob(f'{learmonth}/L25*SRD')

#def month_organization(month_dict):
 #   month_dict = {}
  #  for month in month_dict:
   ##    month_dict[month] = glob.glob(f'{learmonth}/L25{month}*SRD')
    #return month
    
months = ['01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12']
month_dict = {}

for month in months:
    string = ""
    month_dict[month] = glob.glob(f'{learmonth}/L25{month}*SRD')

day_path = month_dict['12'][0]
df = load_Learmonth_data(day_path)
    
#Filtering process:

def filtering(df):
    '''
    This function takes in two arguments, median and result.  Median takes in the Learmonth data
    and plucks out the days that have good data.  This means days that have some form of flux and then
    calculates the median of that certian day, making sure to leave out the bad days of data.'''
    
    median = (not np.isnan(np.nanmedian(df['410']))) and np.nanmedian(df['410']) !=1 and np.nanmedian(df['410'])
    result = median
    return result

good_dat = []
bad_dat = []
   
for month in months:
    for i in trange(len(month_dict[month])):
        path = month_dict[month][i]
        df = load_Learmonth_data(path)
           
        if filtering(df):
            good_dat.append(path)
        else:
            bad_dat.append(path)
            


100%|██████████| 31/31 [02:33<00:00,  4.95s/it]


In [3]:

            
def date_values(good_dat, bad_dat):
    '''
    There are 2 arguments present in this function:
        good_dat: list of data that was made above, that holds all the good data that was filtered out of 
        the original list
        bad_dat: list of the data that filters out all the days that have only NaNs in their system.
    This function's purpose is to change these datetime values in the lists above into usable numbers 
    for interpoliation reasons'''
    
    good_date = []
    bad_date = []
    
    for date in good_dat:
        base = os.path.basename(date)[3:7]
        num_date = int(base)
        good_date.append(num_date)
    
    for nodate in bad_dat:
        nobase = os.path.basename(nodate)[3:7]
        no_num_date = int(nobase)
        bad_date.append(no_num_date)
    
    return good_date, bad_date
        
def time_of_date(good_dat, bad_dat):
    '''
    This function also has two arguments: good_dat, and bad_dat once again.  THe purpoce of this funcion
    is to convert the previous lists into a strtime form of L%y%m%d.SRD so it reconizes that form and 
    actually gives us numbers to work with.'''
    
    good_datetime = []
    bad_datetimes_NaN = []
    
    for good in good_dat:
        file = os.path.basename(good)
        new_var = datetime.strptime(file, "L%y%m%d.SRD")
        good_datetime.append(new_var)
        
    for bad in bad_dat:
        fileb = os.path.basename(bad)
        bad_var = datetime.strptime(fileb, "L%y%m%d.SRD")
        bad_datetimes_NaN.append(bad_var)
        
    return good_datetime, bad_datetimes_NaN


In [4]:

nonexistant_days = [datetime.strptime("L250119.SRD", "L%y%m%d.SRD"),
                    datetime.strptime("L250120.SRD", "L%y%m%d.SRD"),
                    datetime.strptime("L250719.SRD", "L%y%m%d.SRD"),
                    datetime.strptime("L250120.SRD", "L%y%m%d.SRD"),
                    datetime.strptime("L250830.SRD", "L%y%m%d.SRD"),
                    datetime.strptime("L250831.SRD", "L%y%m%d.SRD"),
                    datetime.strptime("L250901.SRD", "L%y%m%d.SRD"),]
nonexistant_days.sort()

def flux_calibration(good_dat):
    '''
    This function is taking the data in good_dat and extracting the median of each individual day that exist for a 
    solar flux value through a loop.  Then afterwards, converting them from unitless counts into units of Janskys, 
    placing them into another list known as flux_Jy.'''
    
    raw_flux = []
    
    for i in trange(len(good_dat)):
        p = good_dat[i]
        df = load_Learmonth_data(p)
        raw_flux.append(np.nanmedian(df['410']))
    
    flux_Jy = [x * 10000 for x in raw_flux]
    return flux_Jy


In [5]:

good_date, bad_date = date_values(good_dat, bad_dat)
good_datetime, bad_datetimes_NaN = time_of_date(good_dat, bad_dat)
flux_Jy = flux_calibration(good_dat)


bad_datetimes_total = sorted(bad_datetimes_NaN + nonexistant_days)
dates = sorted(good_date + bad_date)


  0%|          | 0/342 [00:00<?, ?it/s]

100%|██████████| 342/342 [27:31<00:00,  4.83s/it]


In [6]:

#Data sorting: 

known_jan_points = list(range(101, 132))
known_feb_points = list(range(201, 229))
known_mar_points = list(range(301, 332))
known_apr_points = list(range(401, 431))
known_may_points = list(range(501, 532))
known_jun_points = list(range(601, 631))
known_jul_points = list(range(701, 732))
known_aug_points = list(range(801, 832))
known_sept_points = list(range(901, 931))
known_oct_points = list(range(1001, 1131))
known_nov_points = list(range(1101, 1131))
known_dec_points = list(range(1201, 1232))

real1 = dates[0:29]      #Jan
real2 = dates[30:47]
real3 = dates[48:79]
real4 = dates[80:110]
real5 = dates[111:142]
real6 = dates[143:178]
real7 = dates[179:208]   #July
real8 = dates[207:237]   #August
real9 = dates[237:266]   #Sept
real10 = dates[265:296]
real11 = dates[295:325]
real12 = dates[325:365]

dont_existJ = [item for item in known_jan_points if item not in real1]
dont_existJu = [item for item in known_jul_points if item not in real7]
dont_existaug = [item for item in known_aug_points if item not in real8]
dont_existsep = [item for item in known_sept_points if item not in real9]

dont_exist = sorted(dont_existJ + dont_existJu + dont_existaug + dont_existsep)
bad_dates_total = sorted(bad_date + dont_exist)


In [7]:

#interpolating:

f = interpolate.interp1d((good_date), (flux_Jy))

good_thing = f(good_date)
est_flux = f(bad_dates_total)

for num in bad_dates_total:
    date_str = f"{int(num):04d}"
    full_date_str = f"2025{date_str}"
    
    dt_obj = datetime.strptime(full_date_str, "%Y%m%d")
    

In [8]:

#Pandas dataframes:

good_flux_data = {
    "Date": good_datetime,
    "Flux": flux_Jy,
    "Source": "Real"
}

bad_flux_data = {
    "Date": bad_datetimes_total,
    "Flux": est_flux,
    "Source": "Interpoliated"
}

good = pd.DataFrame(good_flux_data)
bad = pd.DataFrame(bad_flux_data)

total = [good, bad]
full_list = pd.concat(total)
full_list_sort = full_list.sort_values(by=["Date"])

full_list_sort["Date"] == str

full_list_sort.to_csv("testing_flux_data.csv", index=False)

In [9]:
print(bad_date)

[118, 824, 1014, 1015, 1016, 1017, 1018, 1019, 1020, 1021, 1022, 1023, 1024, 1025, 1026, 1027]


In [10]:
type(bad_dat)

list